# Next-word prediction with an LSTM

A small Keras model that learns to predict the next word of a sentence from a single training paragraph. Pipeline: tokenize, build n-gram sequences, pre-pad, train Embedding + LSTM + softmax, generate text.

In [1]:
faqs="""I love to learn machine learning because it helps computers learn from data. Machine learning is a part of artificial intelligence, and deep learning is a part of machine learning. I love to learn deep learning too, because deep learning uses neural networks to solve hard problems. Neural networks learn patterns from data, and the more data they see, the better they become. A neural network has layers, and each layer learns something new from the data. An LSTM is a type of neural network that can remember information for a long time. An LSTM has a forget gate, an input gate, and an output gate. The forget gate decides what information to remove from the cell state. The input gate decides what new information to add to the cell state. The output gate decides what information to send to the next step. I use an LSTM to predict the next word in a sentence. The next word depends on the previous words in the sentence. If the model sees the words I love to, it can predict the next word learn. If the model sees the words deep learning uses, it can predict the next word neural. Every day I learn something new about neural networks and deep learning. Some days I read about machine learning, and some days I build projects with deep learning. I love to build projects because projects help me understand how models really work. My first project was a simple neural network that learned to classify images. My second project was an LSTM that learned to predict the next word. Building projects is hard at first, but practice makes me better every day. When the model makes a mistake, I try to understand why the model made that mistake. Sometimes the model needs more data, and sometimes the model needs more training. Sometimes the model learns too much from the training data and does not work well on new data. This problem is called overfitting, and it is common in machine learning. To reduce overfitting, I can use more data, dropout, or a smaller model. I love to learn how to make my models better. Machine learning is not magic, it is math, data, and practice. Deep learning is powerful, but it needs a lot of data and a lot of computing power. One day I want to build a large model that can understand language like a human. Until then, I will keep learning, keep building, and keep practicing machine learning every day."""

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

In [3]:
tokenizer=Tokenizer()

In [4]:
tokenizer.fit_on_texts([faqs])

In [5]:
tokenizer.word_index

{'the': 1,
 'to': 2,
 'learning': 3,
 'a': 4,
 'i': 5,
 'and': 6,
 'data': 7,
 'is': 8,
 'model': 9,
 'learn': 10,
 'machine': 11,
 'deep': 12,
 'neural': 13,
 'it': 14,
 'an': 15,
 'gate': 16,
 'next': 17,
 'love': 18,
 'from': 19,
 'of': 20,
 'that': 21,
 'can': 22,
 'word': 23,
 'more': 24,
 'new': 25,
 'lstm': 26,
 'information': 27,
 'predict': 28,
 'day': 29,
 'projects': 30,
 'because': 31,
 'networks': 32,
 'better': 33,
 'network': 34,
 'decides': 35,
 'what': 36,
 'in': 37,
 'words': 38,
 'every': 39,
 'build': 40,
 'understand': 41,
 'my': 42,
 'sometimes': 43,
 'needs': 44,
 'keep': 45,
 'part': 46,
 'too': 47,
 'uses': 48,
 'hard': 49,
 'they': 50,
 'has': 51,
 'learns': 52,
 'something': 53,
 'forget': 54,
 'input': 55,
 'output': 56,
 'cell': 57,
 'state': 58,
 'use': 59,
 'sentence': 60,
 'on': 61,
 'if': 62,
 'sees': 63,
 'about': 64,
 'some': 65,
 'days': 66,
 'me': 67,
 'how': 68,
 'models': 69,
 'work': 70,
 'first': 71,
 'project': 72,
 'was': 73,
 'learned': 74,
 

In [6]:
input_sequence=[]
for sentence in faqs.split('.'):
    tokenized_sentence=tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1,len(tokenized_sentence)):
        input_sequence.append(tokenized_sentence[:i+1])


In [7]:
input_sequence

[[5, 18],
 [5, 18, 2],
 [5, 18, 2, 10],
 [5, 18, 2, 10, 11],
 [5, 18, 2, 10, 11, 3],
 [5, 18, 2, 10, 11, 3, 31],
 [5, 18, 2, 10, 11, 3, 31, 14],
 [5, 18, 2, 10, 11, 3, 31, 14, 84],
 [5, 18, 2, 10, 11, 3, 31, 14, 84, 85],
 [5, 18, 2, 10, 11, 3, 31, 14, 84, 85, 10],
 [5, 18, 2, 10, 11, 3, 31, 14, 84, 85, 10, 19],
 [5, 18, 2, 10, 11, 3, 31, 14, 84, 85, 10, 19, 7],
 [11, 3],
 [11, 3, 8],
 [11, 3, 8, 4],
 [11, 3, 8, 4, 46],
 [11, 3, 8, 4, 46, 20],
 [11, 3, 8, 4, 46, 20, 86],
 [11, 3, 8, 4, 46, 20, 86, 87],
 [11, 3, 8, 4, 46, 20, 86, 87, 6],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8, 4],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8, 4, 46],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8, 4, 46, 20],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8, 4, 46, 20, 11],
 [11, 3, 8, 4, 46, 20, 86, 87, 6, 12, 3, 8, 4, 46, 20, 11, 3],
 [5, 18],
 [5, 18, 2],
 [5, 18, 2, 10],
 [5, 18, 2, 1

In [8]:
max_len=max([len(x) for x in input_sequence])

In [9]:
padded_input_sequences = pad_sequences(input_sequence, maxlen=max_len, padding='pre')

In [10]:
max_len

18

In [11]:
X=padded_input_sequences[:,:-1]

In [12]:
y=padded_input_sequences[:,-1]

In [13]:
X

array([[  0,   0,   0, ...,   0,   0,   5],
       [  0,   0,   0, ...,   0,   5,  18],
       [  0,   0,   0, ...,   5,  18,   2],
       ...,
       [  0,   0,   0, ...,  45, 146,  11],
       [  0,   0,   0, ..., 146,  11,   3],
       [  0,   0,   0, ...,  11,   3,  39]], shape=(390, 17), dtype=int32)

In [14]:
vocab_size = len(tokenizer.word_index) + 1  # +1 for the padding index 0
vocab_size

147

In [15]:
# Next-word prediction is a classification problem over the vocabulary, so one-hot encode y
y = to_categorical(y, num_classes=vocab_size)

In [16]:
y.shape

(390, 147)

In [17]:
seq_len = max_len - 1  # X has one column fewer than the padded sequences
seq_len

17

In [18]:
model = Sequential()
model.add(Input(shape=(seq_len,)))
model.add(Embedding(vocab_size, 100))
model.add(LSTM(120))
model.add(Dense(vocab_size, activation='softmax'))

In [19]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 17, 100)        │        14,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 120)            │       106,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 147)            │        17,787 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,567 (541.28 KB)

 Trainable params: 138,567 (541.28 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = model.fit(X, y, epochs=100, verbose=0)
print(f"final loss: {history.history['loss'][-1]:.4f}  final accuracy: {history.history['accuracy'][-1]:.4f}")

final loss: 0.1681  final accuracy: 0.9641


In [22]:
def predict_next(text, n_words=1):
    """Append the n most likely next words to `text`."""
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([text])[0]
        seq = pad_sequences([seq], maxlen=seq_len, padding='pre')
        idx = np.argmax(model.predict(seq, verbose=0))
        text += ' ' + tokenizer.index_word[idx]
    return text

In [23]:
predict_next('Machine')

'Machine learning'

In [24]:
for prompt in ['I love to', 'deep learning uses', 'The forget gate', 'Machine learning is']:
    print(predict_next(prompt, 4))

I love to learn how to make


deep learning uses is powerful but it


The forget gate decides what information to


Machine learning is a part of artificial
